<a href="https://colab.research.google.com/github/mariazafran/Herding-Behaviour-Learning/blob/main/2.Sector-analyis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install yfinance
import yfinance as yf

In [7]:
data = yf.download(
    all_tickers,
    start="2018-01-01",
    end="2024-12-31",
    auto_adjust=False
)

# ✅ FIX MultiIndex properly
data = data.loc[:, ('Adj Close')]

print(data.head())
print(data.shape)

NameError: name 'all_tickers' is not defined

In [6]:
sector_dict = {

    'Financials': [
        'HSBA.L','BARC.L','LLOY.L','STAN.L',
        'PRU.L','AV.L','LGEN.L','MNG.L','SDR.L'
    ],

    'Consumer': [
        'TSCO.L','SBRY.L','MKS.L','ABF.L','ULVR.L',
        'JD.L','FRAS.L','NEXT.L','BDEV.L'
    ],

    'Industrials': [
        'SMIN.L','WEIR.L','IMI.L',
        'SPX.L','ROR.L','QQ.L','MGNS.L','KLR.L'
    ],

    'Healthcare': [
        'AZN.L','GSK.L',
        'HLN.L','INDV.L','HIK.L','SN.L'
    ],

    'Energy': [
        'BP.L','SHEL.L',
        'TLW.L','ENQ.L','HBR.L'
    ],

    'Technology': [
        'SGE.L','AUTO.L',
        'SPT.L','KAIN.L','FDM.L','NCC.L'
    ]
}

In [8]:
all_tickers = [t for v in sector_dict.values() for t in v]

data = yf.download(
    all_tickers,
    start="2018-01-01",
    end="2024-12-31"
)

data = data['Close']

/tmp/ipykernel_6123/2888775535.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[**                     5%                       ]  2 of 43 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SPT.L"}}}
[*********************100%***********************]  43 of 43 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['SPT.L', 'INDV.L', 'NEXT.L', 'KAIN.L', 'BDEV.L']: YFTzMissingError('possibly delisted; no timezone found')


In [9]:
# ✅ Remove columns with all NaN (failed tickers)
data = data.dropna(axis=1, how='all')

# ✅ Forward fill remaining gaps
data = data.ffill()

# ✅ Calculate returns
returns = data.pct_change().dropna()

print(returns.shape)
print(returns.head())

(619, 38)
Ticker         ABF.L    AUTO.L      AV.L     AZN.L    BARC.L      BP.L  \
Date                                                                     
2022-07-19  0.029966  0.003061  0.011259  0.012257  0.025937  0.008500   
2022-07-20  0.006599  0.017972 -0.014423 -0.027956 -0.003666  0.007521   
2022-07-21  0.007747  0.024650  0.012067 -0.011357  0.008374 -0.011841   
2022-07-22  0.004731 -0.004551 -0.006342 -0.002594 -0.007046 -0.001563   
2022-07-25 -0.002060 -0.014370  0.014807  0.004087  0.018246  0.008870   

Ticker         ENQ.L     FDM.L    FRAS.L     GSK.L  ...     SGE.L    SHEL.L  \
Date                                                ...                       
2022-07-19  0.010121  0.011642  0.038647  0.026565  ... -0.004708 -0.001473   
2022-07-20  0.024048  0.016110 -0.003322 -0.004037  ...  0.021881  0.008112   
2022-07-21 -0.025440  0.014722  0.266000 -0.016552  ...  0.018518 -0.010729   
2022-07-22  0.022089  0.021205 -0.007899 -0.002061  ... -0.005114  0.005423 

In [10]:
# get valid tickers from downloaded data
valid_tickers = list(data.columns)

# clean sector dictionary
sector_dict_clean = {
    sector: [t for t in tickers if t in valid_tickers]
    for sector, tickers in sector_dict.items()
}

# check how many firms per sector
for sector, tickers in sector_dict_clean.items():
    print(sector, len(tickers))

Financials 9
Consumer 7
Industrials 8
Healthcare 5
Energy 5
Technology 4


In [11]:
# ✅ Step 5 FINAL FIX

# Remove only fully empty columns (failed tickers)
data = data.dropna(axis=1, how='all')

# Forward fill missing values
data = data.ffill()

# ✅ Calculate returns WITHOUT deleting all rows
returns = data.pct_change()

# ✅ DO NOT drop all NaN rows
# Only remove rows where ALL values are NaN
returns = returns.dropna(how='all')

print(returns.shape)
print(returns.head())

(1766, 38)
Ticker         ABF.L    AUTO.L      AV.L     AZN.L    BARC.L      BP.L  \
Date                                                                     
2018-01-03  0.020871  0.015297  0.000398  0.011949  0.002460  0.012947   
2018-01-04  0.001762 -0.000226  0.011133  0.000968  0.003927  0.011065   
2018-01-05  0.014075 -0.005303 -0.003932  0.006382 -0.026015 -0.000755   
2018-01-08 -0.010410 -0.003648 -0.000790 -0.009800  0.004619 -0.004154   
2018-01-09 -0.003506 -0.007322  0.019755  0.002135  0.008496  0.001137   

Ticker         ENQ.L     FDM.L    FRAS.L     GSK.L  ...     SGE.L    SHEL.L  \
Date                                                ...                       
2018-01-03  0.131757 -0.010834  0.022923  0.016190  ...  0.009396  0.014542   
2018-01-04  0.098506  0.021906 -0.037688 -0.002680  ...  0.005786  0.008561   
2018-01-05  0.005436 -0.009646 -0.010320  0.015975  ...  0.009255 -0.001185   
2018-01-08 -0.008110  0.044372  0.002674 -0.005584  ... -0.005948 -0.001779

In [12]:
# Convert wide → long format
returns_df = returns.reset_index().melt(
    id_vars='Date',
    var_name='Stock',
    value_name='Return'
)

# Create sector mapping (same structure you used before)
sector_map = {ticker: sector for sector, tickers in sector_dict.items() for ticker in tickers}

# Assign sector to each stock
returns_df['Sector'] = returns_df['Stock'].map(sector_map)

# Remove any unmatched
returns_df = returns_df.dropna()

print(returns_df.head())
print(returns_df.shape)

        Date  Stock    Return    Sector
0 2018-01-03  ABF.L  0.020871  Consumer
1 2018-01-04  ABF.L  0.001762  Consumer
2 2018-01-05  ABF.L  0.014075  Consumer
3 2018-01-08  ABF.L -0.010410  Consumer
4 2018-01-09  ABF.L -0.003506  Consumer
(64682, 4)


In [13]:
# ✅ Step 5 — Calculate returns properly

# Remove failed tickers (all NaN columns)
data = data.dropna(axis=1, how='all')

# Forward fill missing values
data = data.ffill()

# Calculate returns WITHOUT deleting everything
returns = data.pct_change()

# Only remove rows where ALL values are missing
returns = returns.dropna(how='all')

print(returns.head())
print(returns.shape)

Ticker         ABF.L    AUTO.L      AV.L     AZN.L    BARC.L      BP.L  \
Date                                                                     
2018-01-03  0.020871  0.015297  0.000398  0.011949  0.002460  0.012947   
2018-01-04  0.001762 -0.000226  0.011133  0.000968  0.003927  0.011065   
2018-01-05  0.014075 -0.005303 -0.003932  0.006382 -0.026015 -0.000755   
2018-01-08 -0.010410 -0.003648 -0.000790 -0.009800  0.004619 -0.004154   
2018-01-09 -0.003506 -0.007322  0.019755  0.002135  0.008496  0.001137   

Ticker         ENQ.L     FDM.L    FRAS.L     GSK.L  ...     SGE.L    SHEL.L  \
Date                                                ...                       
2018-01-03  0.131757 -0.010834  0.022923  0.016190  ...  0.009396  0.014542   
2018-01-04  0.098506  0.021906 -0.037688 -0.002680  ...  0.005786  0.008561   
2018-01-05  0.005436 -0.009646 -0.010320  0.015975  ...  0.009255 -0.001185   
2018-01-08 -0.008110  0.044372  0.002674 -0.005584  ... -0.005948 -0.001779   
2018-01

In [15]:
# Convert wide → long format
returns_df = returns.reset_index().melt(
    id_vars='Date',
    var_name='Stock',
    value_name='Return'
)

# Create sector mapping
sector_map = {ticker: sector for sector, tickers in sector_dict.items() for ticker in tickers}

# Assign sector
returns_df['Sector'] = returns_df['Stock'].map(sector_map)

# Remove any unmatched rows
returns_df = returns_df.dropna()

print(returns_df.head())
print(returns_df.shape)

        Date  Stock    Return    Sector
0 2018-01-03  ABF.L  0.020871  Consumer
1 2018-01-04  ABF.L  0.001762  Consumer
2 2018-01-05  ABF.L  0.014075  Consumer
3 2018-01-08  ABF.L -0.010410  Consumer
4 2018-01-09  ABF.L -0.003506  Consumer
(64682, 4)


In [16]:
# Calculate sector-level market return (Rm)

sector_rm = returns_df.groupby(['Date', 'Sector'])['Return'].mean().reset_index()
sector_rm = sector_rm.rename(columns={'Return': 'Rm'})

print(sector_rm.head())
print(sector_rm.shape)

        Date       Sector        Rm
0 2018-01-03     Consumer  0.015269
1 2018-01-03       Energy  0.050177
2 2018-01-03   Financials -0.000263
3 2018-01-03   Healthcare  0.008672
4 2018-01-03  Industrials  0.007423
(10596, 3)


In [17]:
# Merge firm returns with sector returns
df = returns_df.merge(sector_rm, on=['Date', 'Sector'])

# Compute absolute deviation from sector mean
df['abs_dev'] = abs(df['Return'] - df['Rm'])

# Calculate CSAD (average deviation per sector per day)
csad = df.groupby(['Date', 'Sector']).agg({
    'abs_dev': 'mean',
    'Rm': 'mean'
}).reset_index()

# Rename
csad = csad.rename(columns={'abs_dev': 'CSAD'})

print(csad.head())
print(csad.shape)

        Date       Sector      CSAD        Rm
0 2018-01-03     Consumer  0.008288  0.015269
1 2018-01-03       Energy  0.040790  0.050177
2 2018-01-03   Financials  0.001947 -0.000263
3 2018-01-03   Healthcare  0.005398  0.008672
4 2018-01-03  Industrials  0.007802  0.007423
(10596, 4)


In [18]:
csad['abs_Rm'] = abs(csad['Rm'])
csad['Rm_sq'] = csad['Rm'] ** 2

# Down market dummy
csad['Down'] = (csad['Rm'] < 0).astype(int)

# Volatility (rolling within each sector)
csad['Volatility'] = csad.groupby('Sector')['Rm'].transform(lambda x: x.rolling(20).std())

# Interaction terms
csad['Rm_sq_vol'] = csad['Rm_sq'] * csad['Volatility']
csad['Rm_sq_down'] = csad['Rm_sq'] * csad['Down']

# Drop NaNs from rolling
csad = csad.dropna()

print(csad.head())
print(csad.shape)

          Date       Sector      CSAD        Rm    abs_Rm     Rm_sq  Down  \
114 2018-01-30     Consumer  0.006310 -0.008099  0.008099  0.000066     1   
115 2018-01-30       Energy  0.016133 -0.031629  0.031629  0.001000     1   
116 2018-01-30   Financials  0.005567 -0.017738  0.017738  0.000315     1   
117 2018-01-30   Healthcare  0.002534 -0.009088  0.009088  0.000083     1   
118 2018-01-30  Industrials  0.007711 -0.006493  0.006493  0.000042     1   

     Volatility     Rm_sq_vol  Rm_sq_down  
114    0.009782  6.416329e-07    0.000066  
115    0.020676  2.068461e-05    0.001000  
116    0.006550  2.060800e-06    0.000315  
117    0.010153  8.385052e-07    0.000083  
118    0.008052  3.394882e-07    0.000042  
(10482, 10)


In [19]:
import statsmodels.api as sm

for sector in csad['Sector'].unique():

    sub = csad[csad['Sector'] == sector]

    X = sub[['abs_Rm', 'Rm_sq', 'Volatility', 'Down', 'Rm_sq_vol', 'Rm_sq_down']]
    X = sm.add_constant(X)

    y = sub['CSAD']

    model = sm.OLS(y, X).fit(cov_type='HC3')

    print(f"\n===== {sector} =====")
    print(model.summary())


===== Consumer =====
                            OLS Regression Results                            
Dep. Variable:                   CSAD   R-squared:                       0.389
Model:                            OLS   Adj. R-squared:                  0.387
Method:                 Least Squares   F-statistic:                     50.87
Date:                Sun, 31 May 2026   Prob (F-statistic):           7.34e-58
Time:                        22:33:25   Log-Likelihood:                 6522.3
No. Observations:                1747   AIC:                        -1.303e+04
Df Residuals:                    1740   BIC:                        -1.299e+04
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0036      0.0

In [20]:
# Calculate CSSD (std deviation instead of mean)
cssd = df.groupby(['Date', 'Sector']).agg({
    'Return': 'std',
    'Rm': 'mean'
}).reset_index()

cssd = cssd.rename(columns={'Return': 'CSSD'})

cssd['abs_Rm'] = abs(cssd['Rm'])
cssd['Rm_sq'] = cssd['Rm'] ** 2

print(cssd.head())

        Date       Sector      CSSD        Rm    abs_Rm         Rm_sq
0 2018-01-03     Consumer  0.010124  0.015269  0.015269  2.331450e-04
1 2018-01-03       Energy  0.055938  0.050177  0.050177  2.517773e-03
2 2018-01-03   Financials  0.002351 -0.000263  0.000263  6.940098e-08
3 2018-01-03   Healthcare  0.006615  0.008672  0.008672  7.519629e-05
4 2018-01-03  Industrials  0.010029  0.007423  0.007423  5.510023e-05


In [21]:
for sector in cssd['Sector'].unique():

    sub = cssd[cssd['Sector'] == sector]

    X = sub[['abs_Rm', 'Rm_sq']]
    X = sm.add_constant(X)

    y = sub['CSSD']

    model = sm.OLS(y, X).fit(cov_type='HC3')

    print(f"\n=== CSSD: {sector} ===")
    print(model.summary())


=== CSSD: Consumer ===
                            OLS Regression Results                            
Dep. Variable:                   CSSD   R-squared:                       0.283
Model:                            OLS   Adj. R-squared:                  0.282
Method:                 Least Squares   F-statistic:                     103.5
Date:                Sun, 31 May 2026   Prob (F-statistic):           3.12e-43
Time:                        22:33:33   Log-Likelihood:                 5849.1
No. Observations:                1766   AIC:                        -1.169e+04
Df Residuals:                    1763   BIC:                        -1.168e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0095      0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
csad.to_csv('/content/drive/MyDrive/csad_results.csv', index=False)